# Audio（音频处理）
Audio（音频处理），它本身就可以算是一个模态（单模态），但同时也是多模态系统的一个重要部分（比如「看图听音」）。
## 1. 音频处理常见任务
| 任务类型	| 输入| 	输出| 	典型模型 / 库 |
| --- | --- | --- | --- |
| 自动语音识别 (ASR) |	语音|	文本|	Whisper、Wav2Vec2、HuBERT|
|语音合成 (TTS)|	文本|	语音|	Tacotron 2、VITS、FastSpeech|
|音频分类 / 声音事件检测|	音频|	标签（动物叫声、警报声等）|	AudioSpectrogramTransformer、AST|
|说话人识别 / 验证|	语音|	说话人 ID 或相似度|	ECAPA-TDNN、SpeakerNet|
|音乐相关任务|	音频|	类型分析、节奏识别、分轨等|	MusicGen、Demucs|

## 2. 自动语音识别（ASR）
Automatic Speech Recognition
> 安装必备的基础环境（注意，重启内核）
> - `apt install ffmpeg`
> - `pip install ffmpeg-python`

In [1]:
!pip install ffmpeg-python

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from transformers import pipeline
from IPython.display import Audio

# 
# 如果要识别中文或多语言语音，推荐使用多语言版本 openai/whisper-large-v3
# whisper-tiny：最小、速度快、准确率低
# whisper-base：轻量级
# whisper-small：中等
# whisper-medium：更准
# whisper-large-v3：最准确、需更多显存
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")

audio_file = "audio/audio_2.wav"
Audio(audio_file)

result = asr(audio_file)
print("检测语言并识别文本：", result["text"])

D:\Program Files\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.


检测语言并识别文本： 原因是有人愿为该村电资四八万元费用修桥


### 3. 文本转语音（TTS）
> 安装可能所需的依赖: `pip install sentencepiece soundfile`
> - SentencePiece 是一个文本预处理和分词库，主要用于将文本转换为模型可以理解的格式。
> - SoundFile 是一个音频文件读写库，基于 libsndfile 构建。


In [ ]:
from transformers import pipeline
from IPython.display import Audio


tts = pipeline("text-to-speech", "suno/bark")

speech = tts("SentencePiece 是一个文本预处理和分词库，主要用于将文本转换为模型可以理解的格式. 啊啊啊我不行了", forward_params={"do_sample": True})
print(speech)

Audio(speech['audio'], rate=speech['sampling_rate'])



D:\Program Files\Python312\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
D:\Program Files\Python312\Lib\site-packages\transformers\models\encodec\modeling_encodec.py:120: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("padding_total", torch.tensor(kernel_size - stride, dtype=torch.int64), persistent=False)
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.


### 4. 文本转音频（TTA）
> 根据文本提示生成音乐

In [ ]:

from transformers import pipeline
import scipy

import torch
import gc

# 清空 CUDA 缓存
torch.cuda.empty_cache()

# 强制垃圾回收
gc.collect()

synthesiser = pipeline("text-to-audio", "facebook/musicgen-large")

music = synthesiser("lo-fi music with a soothing melody", forward_params={"do_sample": True})

# 保存
scipy.io.wavfile.write("musicgen_out.wav", rate=music["sampling_rate"], data=music["audio"])

from IPython.display import Audio
Audio("musicgen_out.wav")